# 01 — World Cup Cleaning & Quality Audit (public-data edition)

This notebook reconstructs the match, tournament, and descriptive team-year tables from **OpenFootball only**. It does not redistribute or require the Kaggle dataset, whose licence has not been verified. Match-level observations and aggregate rows must never be concatenated.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RAW = ROOT / 'data/raw/openfootball'
PROC = ROOT / 'data/processed'
REPORTS = ROOT / 'data/reports'
PROC.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
print('JSON tournament files:', len(list(RAW.glob('????/worldcup.json'))))


In [ ]:
import pandas as pd
from IPython.display import display
from src.worldcup_data import load_openfootball
from src.openfootball_summaries import rebuild_tournament_summary, derive_team_year_results
matches = load_openfootball(RAW)
summary = rebuild_tournament_summary(RAW)
team_year = derive_team_year_results(matches)
display(matches.head(), summary.tail(), team_year.head())


## Match data audit

Ambiguous knockout flat-score records are excluded from 90-minute modelling. Tournament champions are computed separately from knockout extra time / penalties.


In [ ]:
print('Matches:', len(matches), 'Eligible:', int(matches.ml_eligible.sum()))
print('Duplicate match keys:', int(matches.duplicated('match_key').sum()))
print('Missing score values:', matches[['goals_1','goals_2']].isna().sum().to_dict())
print('Ambiguous flat knockout:', int(matches.ambiguous_flat_knockout.sum()))
assert not matches.duplicated('match_key').any()
assert len(summary)==23
assert summary.loc[summary.year.eq(2026), 'champion'].iat[0]=='Spain'
assert summary['champion'].replace({'West Germany':'Germany'}).nunique()==8


## Save derived datasets

`team_standings_clean.csv` is intentionally replaced by a different, explicit `team_year_results_90.csv` to avoid calling the 90-minute statistics FIFA official standings.


In [ ]:
matches.to_csv(PROC/'matches_clean.csv', index=False, date_format='%Y-%m-%d', encoding='utf-8-sig')
summary.to_csv(PROC/'tournament_summary_clean.csv', index=False, encoding='utf-8-sig')
team_year.to_csv(PROC/'team_year_results_90.csv', index=False, encoding='utf-8-sig')
coverage = pd.DataFrame({
    'match_records': matches.groupby('year').size(),
    'ml_eligible_matches': matches.groupby('year').ml_eligible.sum(),
    'countries_in_match_records': team_year.groupby('year').team.nunique(),
})
coverage.to_csv(REPORTS/'dataset_coverage.csv')
print('Wrote', list(PROC.glob('*.csv')))
